# Reranker fine-tuning - Domain Adaptive Retrieval

This thin Colab frontend imports the **same package code** used by the CLI and tests. Configure any of the five supported domains in the single configuration cell. All model training and neural inference require **NVIDIA T4 CUDA:0**; CPU is permitted only for dataset preparation and metrics.

**Before running:** Colab > Runtime > Change runtime type > **T4 GPU**. Upload the repository ZIP when prompted. The notebook can download public model weights and datasets, so Internet access is required. CPU-only synthetic checks are available with `dar smoke --offline`. The final scores must be measured; none are pre-filled.

In [ ]:
import torch
assert torch.cuda.is_available(), 'This notebook requires a T4 GPU: Runtime > Change runtime type > T4 GPU.'
assert torch.cuda.device_count() >= 1
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), '| VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('CUDA matrix:', (torch.ones((2, 2), device='cuda:0') @ torch.ones((2, 2), device='cuda:0')).device)


## 1. Provide the repository
Upload the `domain-adaptive-retrieval.zip` generated from this project. Files are extracted into `/content/domain-adaptive-retrieval`. You can skip upload if the repo is already there.

In [ ]:
# Run from a Colab session containing the repository or upload its ZIP file.
# This cell intentionally does not rely on a published GitHub URL or PyPI release.
import os, zipfile
from pathlib import Path
root = Path('/content/domain-adaptive-retrieval')
if not (root / 'pyproject.toml').exists():
    from google.colab import files
    uploaded = files.upload()  # Choose domain-adaptive-retrieval.zip
    zips = [name for name in uploaded if name.endswith('.zip')]
    if not zips:
        raise FileNotFoundError('Upload the complete repository ZIP, not an individual notebook')
    with zipfile.ZipFile(zips[0]) as archive:
        archive.extractall('/content')
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError(f'Expected repository at {root}')
os.chdir(root)
print('Repository:', os.getcwd())


## 2. Install project dependencies
Check CUDA availability again afterward. The GPU runtime has already been initialized; a conflicting Torch installation should be corrected rather than allowing silent CPU fallback.

In [ ]:
# Preserve Colab's working CUDA-enabled PyTorch when it already satisfies the constraints.
# Inspect torch versions after installation to make sure the runtime still has CUDA.
%pip -q install -e '.[train,benchmark]'


In [ ]:
from pathlib import Path
import torch
from domain_adaptive_retrieval.config.loader import load_config
from domain_adaptive_retrieval.models.device import check_device

DOMAIN = 'finance'  # finance | biomedical | legal | ecommerce | code
CFG = load_config(f'configs/domains/{DOMAIN}.yaml', profile='configs/experiments/t4.yaml')
assert CFG.device == 'cuda:0'  # Important: never use the unindexed 'cuda' string here.
print(check_device(CFG.device, require_gpu=True))
# To do a quick subset run first, use:
# CFG.max_train_pairs = 1000
# CFG.max_val_queries = 50
# CFG.max_test_queries = 50
OUTPUT = Path('outputs') / DOMAIN


## 3. Fine-tune and reload on CUDA

In [ ]:
from domain_adaptive_retrieval.workflows.train_reranker import train_reranker
from domain_adaptive_retrieval.models.reranker import RerankerModel

RERANKER_PATH = train_reranker(CFG, OUTPUT / 'reranker')
print('Fine-tuned reranker:', RERANKER_PATH)
loaded = RerankerModel(RERANKER_PATH, device='cuda:0')
print('Reranker CUDA scores:', loaded.predict([('financial stock', 'stock markets')]))


## 4. Evaluate against the frozen base model

In [ ]:
# Frozen base-retriever candidate list, identical for base and tuned rerankers.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'domain_adaptive_retrieval.cli', 'evaluate', 'reranker',
    '--checkpoint', str(RERANKER_PATH), '--domain', DOMAIN, '--profile', 't4',
    '--output', str(OUTPUT / 'reranker' / 'heldout_evaluation'),
], check=True)


## 5. Complete benchmark and publishing
The seven-system matrix is available only after **both** fine-tuned checkpoints exist. Hugging Face publication is a separate explicit CLI action; review source model and dataset licenses beforehand.

In [ ]:
from domain_adaptive_retrieval.workflows.benchmark import run_benchmark
EMBEDDER_PATH = OUTPUT / 'embedding' / 'model'
if EMBEDDER_PATH.is_dir():
    result = run_benchmark(CFG, EMBEDDER_PATH, RERANKER_PATH, OUTPUT / 'benchmark')
    print('Saved S0–S6 metrics and predictions to:', OUTPUT / 'benchmark')
else:
    print('Embedding checkpoint missing. Train embeddings first for complete S0–S6.')
# Publishing is disabled until you run dar publish model --confirm.
